In [1]:
import sys
print(sys.executable)


/Users/blaizelahman/.pyenv/versions/3.10.5/bin/python3.10


In [2]:
import pandas as pd
import numpy as np
import os
import time
import glob
import re

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import BaggingRegressor

import joblib
import nbimporter

from p1_DataImportAndWrangle import customRollingSum

from p4_AdvancedStats import predictMissing

from p6_CurrentSeasonModel import getPreferredLine, grabUpcomingWeekData, predictUpcomingWeek, printPredictions, grabUpcomingYearTalent, grabUpcomingYearSP, grabLastWeekData, savePredictions 

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category = ConvergenceWarning)

pd.set_option('display.max_columns', 1000)
pd.set_option('display.max_rows', 1000)

In [3]:
directory = '/users/blaizelahman/Development/techprojects/CFB Model/2024 Season/Week 16'
pattern = os.path.join(directory, '*_2024week16*.csv')

teamFiles = glob.glob(pattern)

teamDict = {}

for file in teamFiles:

    teamDF = pd.read_csv(file)

    if teamDF.shape[0] >= 56:
    
        key = teamDF['School'][0]

        if key == 'Appalachian State':
            key = 'App State'
        elif key == 'Louisiana Monroe':
            key = 'UL Monroe'
        elif key == 'Connecticut':
            key = 'UConn'
        elif key == 'UT San Antonio':
            key = 'UTSA'
        elif key == 'UMass':
            key = 'Massachusetts'
        elif key == 'Southern Mississippi':
            key = 'Southern Miss'
            
        teamDict[key] = teamDF
    
        print(f'Added: {key}')

Added: New Mexico State
Added: Kansas State
Added: Western Michigan
Added: Florida State
Added: Old Dominion
Added: Stanford
Added: Ole Miss
Added: Memphis
Added: Akron
Added: UConn
Added: Texas A&M
Added: Ohio State
Added: NC State
Added: Massachusetts
Added: Michigan
Added: Tennessee
Added: Oklahoma
Added: Buffalo
Added: Penn State
Added: New Mexico
Added: Arkansas
Added: Nevada
Added: Wyoming
Added: Oregon State
Added: Colorado
Added: Maryland
Added: Idaho
Added: Kansas
Added: Ohio
Added: Washington
Added: Louisiana Tech
Added: Syracuse
Added: Northern Illinois
Added: North Texas
Added: Purdue
Added: Marshall
Added: Missouri
Added: Middle Tennessee
Added: Utah State
Added: UCF
Added: UL Monroe
Added: Cincinnati
Added: Nebraska
Added: Northwestern
Added: SMU
Added: South Florida
Added: Hawai'i
Added: Louisiana
Added: Georgia Tech
Added: Georgia
Added: Temple
Added: Tulane
Added: Wake Forest
Added: Iowa State
Added: Alabama
Added: Virginia
Added: Texas State
Added: Georgia State
Added

In [4]:
directory = '/users/blaizelahman/Development/techprojects/CFB Model/Team Models'
pattern = os.path.join(directory, '*2024*.pkl')

teamFiles = glob.glob(pattern)

modelDict = {}

for file in teamFiles:

    baseName = os.path.basename(file)
    
    key = baseName.split('_model')[0]
    key = key.replace('_', ' ')
    
    # Load your DataFrame
    modelDict[key] = joblib.load(file)

    print(f'Added: {key}')


Added: Georgia State
Added: Florida International
Added: Bowling Green
Added: Virginia
Added: Texas State
Added: Rutgers
Added: Michigan State
Added: Iowa
Added: BYU
Added: West Virginia
Added: East Carolina
Added: SMU
Added: Northwestern
Added: Louisiana
Added: Hawai'i
Added: South Florida
Added: Georgia


/Users/blaizelahman/.pyenv/versions/3.10.5/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.2.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/blaizelahman/.pyenv/versions/3.10.5/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/blaizelahman/.pyenv/versions/3.10.5/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator L

Added: Georgia Tech
Added: Tulane
Added: Wake Forest
Added: Iowa State
Added: Alabama
Added: Temple
Added: Marshall
Added: Utah State
Added: Middle Tennessee
Added: Missouri
Added: UCF
Added: UL Monroe
Added: Cincinnati
Added: Nebraska
Added: Kansas
Added: Ohio
Added: North Texas
Added: Northern Illinois
Added: Louisiana Tech
Added: Syracuse
Added: Washington
Added: Purdue
Added: Arkansas
Added: Nevada
Added: Wyoming
Added: New Mexico
Added: Colorado
Added: Oregon State
Added: Idaho
Added: Maryland
Added: Ohio State
Added: Massachusetts
Added: NC State
Added: Michigan
Added: Tennessee
Added: Penn State
Added: Buffalo
Added: Oklahoma
Added: Kansas State
Added: Western Michigan
Added: Ole Miss
Added: Stanford
Added: Florida State
Added: Old Dominion
Added: Akron
Added: UConn
Added: Memphis
Added: Texas A&M
Added: New Mexico State
Added: Rice
Added: Arizona
Added: LSU
Added: TCU
Added: Central Michigan
Added: Arkansas State
Added: Oklahoma State
Added: Auburn
Added: Wisconsin
Added: San J

In [5]:
directory = '/users/blaizelahman/Desktop/CFB Model/2024 Season/Week 15 FCS'
pattern = os.path.join(directory, '*_2024week15*.csv')

teamFiles = glob.glob(pattern)

fcsDict = {}

for file in teamFiles:

    teamDF = pd.read_csv(file)

    if teamDF.shape[0] >= 10:
    
        key = teamDF['School'][0]
        fcsDict[key] = teamDF
    
        print(f'Added: {key}')

In [6]:
def grabUpcomingWeekData(year, week, teamDict, fcsDict):

    # setting the download directory and Chrome settings
    directory = '/Users/blaizelahman/Development/techprojects/CFB Model'
    chromeOptions = Options()
    prefs = {'download.default_directory': directory}
    chromeOptions.add_experimental_option('prefs', prefs)
    
    # these extra steps are due to the current version of chromedriver not being compatible with the 
    # current version of chrome at the time of development, feel free to delete or change as it pertains
    # to user situation
    path = '/Users/blaizelahman/Downloads/chromedriver_104'
    olderChromePath = '/Applications/Older Chrome.app/Contents/MacOS/Google Chrome'

    chromeOptions.binary_location = olderChromePath
    
    # creating Chrome driver
    driver = webdriver.Chrome(service = Service(path), options = chromeOptions)

    # setting the link that directs to the betting data
    bettingDataLink = f'https://collegefootballdata.com/exporter/lines?year={year}&week={week}&seasonType=regular'
    
    driver.get(bettingDataLink)
    time.sleep(4) 
            
    # clicking the query button
    query = driver.find_element(By.XPATH, "//button[contains(span/text(), 'Query')]")
    query.click()
    time.sleep(3) 
            
    # clicking the export button
    export = driver.find_element(By.XPATH, "//button[contains(span/text(), 'Export')]")
    export.click()
    time.sleep(3)

    key = str(year)
            
    # grabs files from CFBData folder
    files = os.listdir(directory)
        
    # grab the file paths for all files ending in .csv
    filePaths = [os.path.join(directory, name) for name in files if name.endswith('.csv')]

    # grabbing the most recently made file out of those in paths
    file = max(filePaths, key=os.path.getctime)
            
    # loading csv file
    bettingData = pd.read_csv(file)

    # deleting the file after it has been added
    os.remove(file)

    driver.quit()

    # setting our preferred line providers
    lineProviders = ['ESPN Bet', 'DraftKings', 'consensus', 'Bovada']

    # getting the preferred line for each individual game
    preferredLines = bettingData.groupby('Id').apply(lambda x: getPreferredLine(x, lineProviders)).reset_index(drop = True)

    # adding a new row to each home team's dataframe with betting data and both teams' rolling sum data
    for index, row in preferredLines.iterrows():
    
        # grabbing the home team's name if it's in teamDict
        if row['HomeTeam'] in teamDict:
            homeTeam = row['HomeTeam']

        # grabbing if home team's name if it's in fcsDict
        elif row['HomeTeam'] in fcsDict:
            homeTeam = row['HomeTeam']
            
        else: 
            homeTeam = None

        # grabbing the away team's name if it's in teamDict
        if row['AwayTeam'] in teamDict: 
            awayTeam = row['AwayTeam']

        # grabbing if home team's name if it's in fcsDict
        elif row['AwayTeam'] in fcsDict:
            awayTeam = row['AwayTeam']
            
        else:
            awayTeam = None

        rowDF = pd.DataFrame([row])

        # adding new row with betting data for the home team if they're in teamDict
        if homeTeam != None:

            # checking if team is in teamDict and adding new row if so
            if homeTeam in teamDict:

                # correcting the spread format
                if row['HomeTeam'] == homeTeam:
                    rowDF.iloc[0, rowDF.columns.get_loc('Spread')] = float(rowDF.iloc[0, rowDF.columns.get_loc('Spread')]) * -1
                
                # grabbing the teams' dataframes and adding a new row with the betting data
                homeDF = teamDict[homeTeam] 
                homeDF = pd.concat([homeDF, rowDF], ignore_index = True)
                homeDF.reset_index(drop = True, inplace = True)
                teamDict[homeTeam] = homeDF
    

            # adding new row to FCS team if team isn't in teamDict
            else:

                # correcting the spread format
                if row['HomeTeam'] == homeTeam:
                    rowDF.iloc[0, rowDF.columns.get_loc('Spread')] = float(rowDF.iloc[0, rowDF.columns.get_loc('Spread')]) * -1
                    
                # grabbing the teams' dataframes and adding a new row with the betting data
                homeDF = fcsDict[homeTeam] 
                homeDF = pd.concat([homeDF, rowDF], ignore_index = True)
                homeDF.reset_index(drop = True, inplace = True)
                fcsDict[homeTeam] = homeDF
                

        # adding new row with betting data for the away team if they're in teamDict
        if awayTeam != None:

            # checking if team is in teamDict and adding new row if so
            if awayTeam in teamDict:

                # correcting the spread format
                if row['HomeTeam'] == homeTeam:
                    rowDF.iloc[0, rowDF.columns.get_loc('Spread')] = float(rowDF.iloc[0, rowDF.columns.get_loc('Spread')]) * -1
                
                # grabbing the teams' dataframes and adding a new row with the betting data
                awayDF = teamDict[awayTeam] 
                awayDF = pd.concat([awayDF, rowDF], ignore_index = True)
                awayDF.reset_index(drop = True, inplace = True)
                teamDict[awayTeam] = awayDF

            else:

                # correcting the spread format
                if row['HomeTeam'] == homeTeam:
                    rowDF.iloc[0, rowDF.columns.get_loc('Spread')] = float(rowDF.iloc[0, rowDF.columns.get_loc('Spread')]) * -1
                
                # grabbing the teams' dataframes and adding a new row with the betting data
                awayDF = fcsDict[awayTeam] 
                awayDF = pd.concat([awayDF, rowDF], ignore_index = True)
                awayDF.reset_index(drop = True, inplace = True)
                teamDict[awayTeam] = awayDF
        
                
    # grabbing all of the rolling columns that we will have to update (not including opposing columns)
    rollingCols = [col for col in teamDict['Florida State'].columns if 'rolling_sum' in col and '_opp' not in col]

    # grabbing rolling sum data for teams in teamDict
    for team, teamDF in teamDict.items():

        # checking if a new row has been initialized for the upcoming week and, if it hasn't,
        # say that the team isn't playing this week and skip it
        if pd.isna(teamDF.iloc[-1]['School']) == False:
            print(f'{team} not playing this week')
            continue
    
        for col in rollingCols:
            
            # grabbing the column and window of games to pull from
            parts = col.split('_')
            columnName = parts[2]

            if columnName[-1] == '8':
                columnName = columnName[:-1]
                window = 8
            else:
                columnName = columnName[:-2]
                window = 20
            
            # grabbing the rolling sum and setting the given column in the most recent row with it
            rollingSum = customRollingSum(teamDF[columnName], window)

            # correcting yardsPerPass and yardPerRushAttempt columns as we did in p1
            if col == 'rolling_sum_yardsPerPass20' or col == 'rolling_sum_yardsPerRushAttempt20':
                rollingSum = rollingSum / 20

            if col == 'rolling_sum_yardsPerPass8' or col == 'rolling_sum_yardsPerRushAttempt8':
                rollingSum = rollingSum / 8
                
            teamDF.at[teamDF.index[-1], col] = rollingSum.iloc[-1]

    # grabbing rolling sum data for teams in fcsDict
    for team, teamDF in fcsDict.items():

        # checking if a new row has been initialized for the upcoming week and, if it hasn't,
        # say that the team isn't playing this week and skip it
        if pd.isna(teamDF.iloc[-1]['School']) == False and team not in teamDict:
            continue

        
        for col in rollingCols:
            
            # grabbing the column and window of games to pull from
            parts = col.split('_')
            columnName = parts[2]

            if columnName[-1] == '8':
                columnName = columnName[:-1]
                window = 8
            else:
                columnName = columnName[:-2]
                window = 20
            
            # grabbing the rolling sum and setting the given column in the most recent row with it
            rollingSum = customRollingSum(teamDF[columnName], window)

            # correcting yardsPerPass and yardPerRushAttempt columns as we did in p1
            if col == 'rolling_sum_yardsPerPass20' or col == 'rolling_sum_yardsPerRushAttempt20':
                rollingSum = rollingSum / 20

            if col == 'rolling_sum_yardsPerPass8' or col == 'rolling_sum_yardsPerRushAttempt8':
                rollingSum = rollingSum / 8

            
            teamDF.at[teamDF.index[-1], col] = rollingSum.iloc[-1]

    # creating Chrome driver
    driver = webdriver.Chrome(service = Service(path), options = chromeOptions)

    # setting the link that directs to the betting data
    spRatingsLink = f'https://collegefootballdata.com/exporter/ratings/sp?year={year}'
    
    driver.get(spRatingsLink)
    time.sleep(4) 
            
    # clicking the query button
    query = driver.find_element(By.XPATH, "//button[contains(span/text(), 'Query')]")
    query.click()
    time.sleep(3) 
            
    # clicking the export button
    export = driver.find_element(By.XPATH, "//button[contains(span/text(), 'Export')]")
    export.click()
    time.sleep(3)

    key = str(year)
            
    # grabs files from CFBData folder
    files = os.listdir(directory)
        
    # grab the file paths for all files ending in .csv
    filePaths = [os.path.join(directory, name) for name in files if name.endswith('.csv')]

    # grabbing the most recently made file out of those in paths
    file = max(filePaths, key=os.path.getctime)
            
    # loading csv file
    spRatings = pd.read_csv(file)

    os.remove(file)

    driver.quit()

    name = f'SP_{year}week{week}.csv'
    path = os.path.join(f'/users/blaizelahman/Development/techprojects/CFB Model/SP+ Data/{year}', name)
    spRatings.to_csv(path)
    print('CSV: ' + name)

    # importing in team talent data gathered from collegefootballdata.com
    path = f'/Users/blaizelahman/Development/techprojects/CFB Model/Talent Data/talent_{year}.csv'
    talentRatings = pd.read_csv(path)

    # modified version of the 'mergeRollingSum' function from p1 that will grab opponent rolling_sum
    # data for each game, will also grab SP+ and talent data from past week (unless it's week 1,
    # which will be skipped as a different function will be used to grab new data for the upcoming year)
    for team, teamDF in teamDict.items():

        # checking if a new row has been initialized for the upcoming week and, if it hasn't,
        # skip it
        if pd.isna(teamDF.iloc[-1]['School']) == False:
            continue
        
        teamDF = teamDict[team].copy()

        # grabbing most recent game and index
        lastRow = teamDF.iloc[-1]
        lastIndex = teamDF.index[-1]
    
        # grabbing the game id and opponent's name to access the game in their dataframe
        gameID = lastRow['Id']

        if team == lastRow['HomeTeam']:
            oppName = lastRow['AwayTeam']
        else:
            oppName = lastRow['HomeTeam']

        
        # grabbing the opponent's dataframe based on what dictionary they're in
        if oppName in teamDict:
            
            oppDF = teamDict[oppName]
            
        elif oppName in fcsDict: 
            oppDF = fcsDict[oppName]

        else: 
            continue

        # grabbing the opponent's rolling_sum columns from the game they played against the given team
        oppRow = oppDF[oppDF['Id'] == gameID]

        if not oppRow.empty:
            
            # updating rolling sum columns for opposing team
            for col in rollingCols:
                    
                teamDF.loc[lastIndex, col + '_opp'] = oppRow.iloc[0][col]

        # skipping the talent and SP+ rating step if it's week 1 because this will be done using
        # grabUpcomingYearTalent and grabUpcomingYearSP
        if week == 1:
            teamDict[team] = teamDF
            continue
            
        # ensure columns exist and set them as NaN values if not
        if 'talent' not in teamDF.columns:
            teamDF['talent'] = pd.NA
        if 'SP' not in teamDF.columns:
            teamDF['SP'] = pd.NA

        # grabbing team talent rating from talentRatings
        talent = talentRatings.loc[talentRatings['School'] == team, 'Talent']
              
        if len(talent.values) == 0:
            print(f'No talent rating for {team}')
            continue

        # assigns talent column with the corresponding talent ratings for games in the given year
        teamDF.loc[lastIndex, 'talent'] = talent.values[0]

        # grabbing team talent rating from talentRatings
        sp = spRatings.loc[spRatings['Team'] == team, 'Rating']
              
        if len(sp.values) == 0:
            continue

        # assigns talent column with the corresponding talent ratings for games in the given year
        teamDF.loc[lastIndex, 'SP'] = sp.values[0]

        teamDict[team] = teamDF
        

    # adding in the opposing team's talent and SP+ rating
    for team, teamDF in teamDict.items():
        
        teamDF = teamDict[team].copy()

        # grabbing most recent game and index
        lastRow = teamDF.iloc[-1]
        lastIndex = teamDF.index[-1]

        if team == lastRow['HomeTeam']:
            oppName = lastRow['AwayTeam']
        else:
            oppName = lastRow['HomeTeam']

        # grabbing the opponent's dataframe based on what dictionary they're in
        if oppName in teamDict:
            
            oppDF = teamDict[oppName]

            # setting if a team is in the FCS or not because the naming conventions for columns differs
            fcs = False
            
        elif oppName in fcsDict and oppName not in teamDict: 
            
            oppDF = fcsDict[oppName]

            # setting if a team is in the FCS or not because the naming conventions for columns differs
            fcs = True

        else: 
            continue

        # skipping the talent and SP+ adding process if it's week 1
        if week == 1:
            teamDict[team] = teamDF
            continue

        # seeing if a team is in the FCS or not because the naming conventions for columns differs
        if fcs == False:
            
            # grabbing the opposing team's most recent played game to grab their talent and SP+ data
            filteredOppDF = oppDF[oppDF['Year'] == year]

        if fcs == True:
            filteredOppDF = oppDF[oppDF['Year'] == str(year)]

        # checking if dataframe is empty
        if filteredOppDF.empty:
            print(f"No data for opponent in year {year}: {oppName}")
            continue 

        oppRow = filteredOppDF.iloc[0]

        # ensure columns exist and set them as NaN values if not
        if 'talent' not in oppRow.index:
            oppRow['talent'] = pd.NA
        if 'SP' not in oppRow.index:
            oppRow['SP'] = pd.NA
            
        # adding oppossing team talent ratings
        teamDF.loc[lastIndex, 'talent' + '_opp'] = oppRow['talent']

        # getting the opponent's talent rating column from the game they played the given team
        oppRow = spRatings[spRatings['Team'] == oppName]
    
        if not oppRow.empty:

            oppSP = oppRow.iloc[0]['Rating']
        
            # merging the opponent's talent column on the row the team plays them
            teamDF.loc[teamDF.index[-1], 'SP_opp'] = oppSP 

        teamDict[team] = teamDF

    return teamDict
        
        

In [7]:
grabUpcomingWeekData(2024, 13, teamDict, fcsDict)

SessionNotCreatedException: Message: session not created: This version of ChromeDriver only supports Chrome version 104
Current browser version is 138.0.7204.184 with binary path /Applications/Older Chrome.app/Contents/MacOS/Google Chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
0   chromedriver_104                    0x00000001045b2ae0 chromedriver_104 + 3828448
1   chromedriver_104                    0x0000000104547f1c chromedriver_104 + 3391260
2   chromedriver_104                    0x0000000104240fcc chromedriver_104 + 217036
3   chromedriver_104                    0x0000000104263a40 chromedriver_104 + 358976
4   chromedriver_104                    0x0000000104260440 chromedriver_104 + 345152
5   chromedriver_104                    0x000000010425d05c chromedriver_104 + 331868
6   chromedriver_104                    0x000000010428e254 chromedriver_104 + 533076
7   c

In [ ]:
teamDict['Maryland'][-5:]

In [ ]:
teamDict['SMU'][-5:]

In [ ]:
teamDict['Nevada'][-10:]

In [ ]:
def grabUpcomingYearTalent(year, teamDict, fcsDict):

    # importing in team talent data gathered from collegefootballdata.com
    path = f'/Users/blaizelahman/Desktop/CFB Model/Talent Data/talent_{year}.csv'
    talentRatings = pd.read_csv(path)

    # grabbing team talent data for FBS teams
    for team, teamDF in teamDict.items():

        teamDF = teamDF.copy()

        # checking that the team is playing in week 1 and skipping any team that isn't
        if pd.isna(teamDF.iloc[-1]['School']) == False:
            continue

        # setting the year for the week 1 row
        teamDF.loc[teamDF.index[-1], 'Year'] = year

        # grabbing team talent rating from talentRatings
        talent = talentRatings.loc[talentRatings['School'] == team, 'Talent']
              
        if len(talent.values) == 0:
            print(f'No talent rating for {team}')
            continue

        # assigns talent column with the corresponding talent ratings for games in the given year
        teamDF.loc[teamDF['Year'] == year, 'talent'] = talent.values[0]

        teamDict[team] = teamDF

    # grabbing team talent data for FCS teams
    for team, teamDF in fcsDict.items():

        teamDF = teamDF.copy()

        # checking that the team is playing in week 1 and skipping any team that isn't
        if pd.isna(teamDF.iloc[-1]['School']) == False:
            continue

        # setting the year for the week 1 row
        teamDF.loc[teamDF.index[-1], 'Year'] = str(year)
        
        # grabbing team talent rating from talentRatings
        talent = talentRatings.loc[talentRatings['School'] == team, 'Talent']

        if len(talent.values) == 0:
            print(f'No talent rating for {team}')
            continue
            
        # assigns talent column with the corresponding talent ratings for games in the given year
        teamDF.loc[teamDF['Year'] == str(year), 'talent'] = talent.values[0]

        fcsDict[team] = teamDF

    # now grabbing the opponent's talent data and updating each team's dataframes with it
    for team, teamDF in teamDict.items():

        teamDF = teamDF.copy()

        lastRow = teamDF.iloc[-1]

        lastRow['talent_opp'] = np.nan

        if team == lastRow['HomeTeam']:
            
            # grabbing each opponent's name to access their talent rating in talentRatings
            oppName = lastRow['AwayTeam']

        if team == lastRow['AwayTeam']:
            
            # grabbing each opponent's name to access their talent rating in talentRatings
            oppName = lastRow['HomeTeam']
            
         
        # getting the opponent's talent rating column from the game they played the given team
        oppRow = talentRatings[talentRatings['School'] == oppName]
    
        if not oppRow.empty:

            oppTalent = oppRow.iloc[0]['Talent']
        
            # merging the opponent's talent column on the row the team plays them
            teamDF.loc[teamDF.index[-1], 'talent_opp'] = oppTalent 

        teamDict[team] = teamDF
        
        

In [8]:
def grabUpcomingYearSP(year, week, teamDict):
    
    # importing in team SP+ data gathered from collegefootballdata.com
    path = f'/Users/blaizelahman/Desktop/CFB Model/SP+ Data/2024/SP_{year}week{week}.csv'
    spRatings = pd.read_csv(path)

    # grabbing team talent data
    for team, teamDF in teamDict.items():

        teamDF = teamDF.copy()

        # checking that the team is playing in week 1 and skipping any team that isn't
        if pd.isna(teamDF.iloc[-1]['School']) == False:
            continue

        # setting the year for the week 1 row
        teamDF.loc[teamDF.index[-1], 'Year'] = year

        # grabbing team talent rating from talentRatings
        sp = spRatings.loc[spRatings['Team'] == team, 'Rating']
              
        if len(sp.values) == 0:
            continue

        # assigns talent column with the corresponding talent ratings for games in the given year
        teamDF.loc[teamDF['Year'] == year, 'SP'] = sp.values[0]

        teamDict[team] = teamDF

    # now grabbing the opponent's talent data and updating each team's dataframes with it
    for team, teamDF in teamDict.items():

        teamDF = teamDF.copy()

        lastRow = teamDF.iloc[-1]

        lastRow['SP_opp'] = np.nan

        if team == lastRow['HomeTeam']:
            
            # grabbing each opponent's name to access their talent rating in talentRatings
            oppName = lastRow['AwayTeam']

        if team == lastRow['AwayTeam']:
            
            # grabbing each opponent's name to access their talent rating in talentRatings
            oppName = lastRow['HomeTeam']
            
         
        # getting the opponent's talent rating column from the game they played the given team
        oppRow = spRatings[spRatings['Team'] == oppName]
    
        if not oppRow.empty:

            oppSP = oppRow.iloc[0]['Rating']
        
            # merging the opponent's talent column on the row the team plays them
            teamDF.loc[teamDF.index[-1], 'SP_opp'] = oppSP 

        teamDict[team] = teamDF

In [ ]:
grabUpcomingYearSP(2024, 2, teamDict)

In [9]:
features = [col for col in teamDict['Florida State'].columns if any(word in col for word in ['rolling_sum','talent','SP'])]

In [10]:
def predictUpcomingWeek(modelDict, teamDict, features):

    predsDict = {}

    sortedModels = {key: modelDict[key] for key in sorted(modelDict)}

    for team, model in sortedModels.items():

        pred = None
        spread = None
        spreadDiff = None
        
        # only looking at teams with a game this week
        if pd.isna(teamDict[team].iloc[-1]['School']) == True:

            # skipping a team if its game has already been predicted
            homeTeam = teamDict[team].iloc[-1]['HomeTeam'] == team

            # checking if the given team is the home team or not
            if homeTeam == True:

                # getting the opposing team's name
                oppTeam = teamDict[team].iloc[-1]['AwayTeam']

                # skipping the prediction if that game has already been predicted by other team
                if oppTeam < team and oppTeam in sortedModels and team != 'Baylor':

                    predsDict[team] = f'Already predicted by {oppTeam}'
                    continue

            else:

                # getting the opposing team's name
                oppTeam = teamDict[team].iloc[-1]['HomeTeam']
                
                # skipping the prediction if that game has already been predicted by other team
                if oppTeam < team and oppTeam in sortedModels:

                    predsDict[team] = f'Already predicted by {oppTeam}'
                    continue
                

            # grabbing feature data and predicting a score differential
            X = teamDict[team].iloc[-1][features].to_frame().T
            pred = model.predict(X)
            pred = round(pred[0] * 2) / 2

            # grabbing the Vegas spread and spread differential
            spread = teamDict[team].iloc[-1]['Spread']
            spreadDiff = pred - spread

            # grabbing the gameID from the game
            gameID = teamDict[team].iloc[-1]['Id']

        else: 
            predsDict[team] = 'No game this week'
            continue

        # creating a list of teams that have incomplete data due to being added to the FBS too recently
        skipList = ['Jacksonville State', 'James Madison', 'Kennesaw State', 
                    'Coastal Carolina', 'Liberty', 'Sam Houston State']

        # skipping predicting teams on the skip list because the predictions are inaccurate due to bad data
        if homeTeam in skipList or oppTeam in skipList:
            predsDict[team] = 'Playing team with incomplete data'
            continue

        cover = np.nan if pd.isna(spreadDiff) else -1 if spreadDiff < 0 else 1 if spreadDiff > 0 else 0

        predsDict[team] = [pred, spread, spreadDiff, cover, gameID, team, oppTeam]

    return predsDict



In [ ]:
predsDict = predictUpcomingWeek(modelDict, teamDict, features)

In [11]:
path = '/users/blaizelahman/Development/techprojects/CFB Model/Bin Data/Bin_Data.csv'
binData = pd.read_csv(path)

In [12]:
def printPredictions(predsDict, binData, teamDict, fcsDict):

    # setting up lists that will hold all bets with their respective teams, whether 
    # or not they will cover, and their distinctions
    bestList = []
    greatList = []
    goodList = []
    normalList = []
    tossUpList = []

    # going through predictions and seeing if our model thinks teams and binning the 
    # spread differentials
    for team, preds in predsDict.items():

        # skipping if no prediction or game already predicted
        if len(preds) != 7:
            print(f'No prediction for {team}.')
            continue

        # skipping if it's an FCS prediction
        if preds[-1] in fcsDict or preds[-1] not in teamDict:
            continue
        
        spreadDiff = preds[2]

        # setting a value for cover and marking whether a team is predicted to cover (1) or not (-1)
        # (0 means toss up)
        cover = 1 if spreadDiff > 0 else -1 if spreadDiff < 0 else 0 

        # adding the prediction to the toss up list and skipping to the next one
        if cover == 0:
            tossUpList.append([team, preds[1], preds[0]])
            continue

        # grabbing the success rate from the bin that spreadDiff belongs to
        predBin = binData[(binData['lowerBin'] <= spreadDiff) & (binData['upperBin'] > spreadDiff)]

        if not predBin.empty:

            successRate = predBin.iloc[0]['successRate']

        else: 
            print(f'No bin found for spreadDiff {spreadDiff} \n')


        # adding the prediction to its respective list based on success rate
        if successRate < 0.595:
            normalList.append([team, cover, preds[0], successRate, preds[1]])
        elif successRate <= 0.645:
            goodList.append([team, cover, preds[0], successRate, preds[1]])
        elif successRate <= 0.695:
            greatList.append([team, cover, preds[0], successRate, preds[1]])
        else: 
            bestList.append([team, cover, preds[0], successRate, preds[1]])

    print()

    # sorting the games by success rate in each list so they print out in 
    # order of best success rate to worst
    normalList.sort(key = lambda x: x[3], reverse = True)
    goodList.sort(key = lambda x: x[3], reverse = True)
    greatList.sort(key = lambda x: x[3], reverse = True)
    bestList.sort(key = lambda x: x[3], reverse = True)

    print('Games with a greater than 70% success rate: \n')
    if len(bestList) == 0:
        print('No games above a 70% success rate this week.')
    else:
        for game in bestList:
            if game[1] == -1:
                print(f'{game[0]}: NOT COVER with spread {game[-1]}. Predicted score differential: ' \
                f'{game[2]}. Historical success rate: {(game[3] * 100):.2f}%.')
  
            else:
                print(f'{game[0]}: COVER with spread {game[-1]}. Predicted score differential of ' \
                f'{game[2]}. Historical success rate {(game[3] * 100):.2f}%.')

    print()
    

    print('Games with a 65-70% success rate: \n')
    if len(greatList) == 0:
        print('No games with a 65-70% success rate this week.')
    else:
        for game in greatList:
            if game[1] == -1:
                print(f'{game[0]}: NOT COVER with spread {game[-1]}. Predicted score differential: ' \
                f'{game[2]}. Historical success rate: {(game[3] * 100):.2f}%.')
  
            else:
                print(f'{game[0]}: COVER with spread {game[-1]}. Predicted score differential of ' \
                f'{game[2]}. Historical success rate {(game[3] * 100):.2f}%.')

    print()

    print('Games with a 60-65% success rate: \n')
    if len(goodList) == 0:
        print('No games with a 60-65% success rate this week.')
    else:
        for game in goodList:
            if game[1] == -1:
                print(f'{game[0]}: NOT COVER with spread {game[-1]}. Predicted score differential: ' \
                f'{game[2]}. Historical success rate: {(game[3] * 100):.2f}%.')
  
            else:
                print(f'{game[0]}: COVER with spread {game[-1]}. Predicted score differential of ' \
                f'{game[2]}. Historical success rate {(game[3] * 100):.2f}%.')

    print()

    print('Games with a less than 60% success rate: \n')
    if len(normalList) == 0:
        print('No games with a lower than 60% success rate this week.')
    else:
        for game in normalList:
            if game[1] == -1:
                print(f'{game[0]}: NOT COVER with spread {game[-1]}. Predicted score differential: ' \
                f'{game[2]}. Historical success rate: {(game[3] * 100):.2f}%.')
  
            else:
                print(f'{game[0]}: COVER with spread {game[-1]}. Predicted score differential of ' \
                f'{game[2]}. Historical success rate {(game[3] * 100):.2f}%.')

    print()
    
    print('Games that are tossups: \n')
    if len(tossUpList) == 0:
        print('No games that are tossups this week.')
    else:
        for game in tossUpList:
            print(f'{game[0]}: TOSS UP with spread {game[1]} and predicted score differential {game[2]}.' \
                  f' Bet at your own discretion.')
                

In [ ]:
predsDict['Maryland']

In [ ]:
predsDict['Florida']

In [ ]:
predsDict['Colorado']

In [13]:
def grabLastWeekData(year, week, teamDict, fcsDict):

    # ignoring a redundant warning message
    warnings.simplefilter(action = 'ignore', category = FutureWarning)

    # setting the download directory and Chrome settings
    directory = '/Users/blaizelahman/Development/techprojects/CFB Model/Recent Data'
    chromeOptions = Options()
    prefs = {'download.default_directory': directory}
    chromeOptions.add_experimental_option('prefs', prefs)
    
    # these extra steps are due to the current version of chromedriver not being compatible with the 
    # current version of chrome at the time of development, feel free to delete or change as it pertains
    # to user situation
    path = '/Users/blaizelahman/Downloads/chromedriver_104'
    olderChromePath = '/Applications/Older Chrome.app/Contents/MacOS/Google Chrome'

    chromeOptions.binary_location = olderChromePath
    
    # creating Chrome driver
    driver = webdriver.Chrome(service = Service(path), options = chromeOptions)

    link = f'https://collegefootballdata.com/exporter/games/teams?year={year}&week={week}&seasonType=regular'

    driver.get(link)
    time.sleep(4) 
            
    # clicking the query button
    query = driver.find_element(By.XPATH, "//button[contains(span/text(), 'Query')]")
    query.click()
    time.sleep(3) 
            
    # clicking the export button
    export = driver.find_element(By.XPATH, "//button[contains(span/text(), 'Export')]")
    export.click()
    time.sleep(3)

    # grabs files from CFBData folder
    files = os.listdir(directory)
        
    # grab the file paths for all files ending in .csv
    filePaths = [os.path.join(directory, name) for name in files if name.endswith('.csv')]

    # grabbing the most recently made file out of those in paths
    file = max(filePaths, key = os.path.getctime)
            
    # loading csv file
    totalWeeklyData = pd.read_csv(file)

    driver.quit()

    # creates new week column that displays the week the game took place
    totalWeeklyData['Week'] = f"Week {week}"

    totalWeeklyData['Year'] = year

    # drops the redundant first row and resets the indices of the datadrame
    totalWeeklyData.drop(0, inplace = True)

    totalWeeklyData.reset_index(drop = True, inplace = True)

    totalWeeklyData.reset_index(level = 0, drop = True, inplace = True)

    totalWeeklyData.rename(columns = {0: 'Game Id', 1: 'School', 2: 'Conference', 
                                  3: 'HomeAway', 4: 'Points', 5: 'Stat Category', 6: 'Stat'}, inplace = True)

    # making a copy of the dataframe to look for non-numeric values
    totalWeeklyDataCopy = totalWeeklyData.copy(deep = True)
    
    # converting string type data points to numerics
    totalWeeklyDataCopy['Stat'] = pd.to_numeric(totalWeeklyDataCopy['Stat'], errors='coerce')
    
    # filter out unique non-numeric rows
    nan_stats = totalWeeklyDataCopy[totalWeeklyDataCopy['Stat'].isna()]['Stat Category'].unique()

    # replacing the "-" in totalPenaltiesYards, completionAttempts, fourthDownEff, and thirdDownEff
    # with ".", and the same with ":" in possessionTime so they can be converted to numeric values
    totalWeeklyData['Stat'] = totalWeeklyData['Stat'].str.replace('-', '.')
    totalWeeklyData['Stat'] = totalWeeklyData['Stat'].str.replace(':', '.')
    
    totalWeeklyData['Stat'] = pd.to_numeric(totalWeeklyData['Stat'], errors = 'coerce')
    
    # converting points to numerics
    totalWeeklyData['Points'] = pd.to_numeric(totalWeeklyData['Points'], errors = 'coerce')

    # pivoting the Stat Category column into multiple columns each with their respective stat
    totalWeeklyData = totalWeeklyData.pivot(index=['Game Id', 'School', 'Conference', 'HomeAway', 'Points', 'Week', 'Year'], 
                          columns='Stat Category', 
                          values='Stat').reset_index()
    
    # flattening the columns
    totalWeeklyData.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in totalWeeklyData.columns]

    tempDict = {}
    for team in totalWeeklyData.School.unique():
        
        dfName = team

        # getting rid of redundant error warning
        pd.options.mode.chained_assignment = None
    
    
        # getting all games with the given team
        teamGames = totalWeeklyData[totalWeeklyData['School'] == team]
        gameIDs = teamGames['Game Id'].unique()
        
        # adding in all opponents of the given team and their stats
        teamDF = totalWeeklyData[totalWeeklyData['Game Id'].isin(gameIDs)]
    
        # converting the week column to an int so the dataframe can then be sorted by year and week
        teamDF['Week'] = teamDF['Week'].str[-2:].astype(int)
        teamDF = teamDF.sort_values(by = ["Year", "Week"])
    
        teamDF = teamDF.reset_index(drop=True)
    
        # adding a total touchdown column
        teamDF['totalTDs'] = teamDF[['passingTDs', 'rushingTDs', 'interceptionTDs', 'kickReturnTDs', 'puntReturnTDs']].sum(axis = 1, skipna = True)
    
        # merging dataframe to pair teams who played each other by Game Id and differentiating the opponent's stats
        mergeTeamDF = teamDF.merge(teamDF, on='Game Id', suffixes=('', '_opp'))
    
        # making sure there's no duplicates
        mergeTeamDF = mergeTeamDF[mergeTeamDF['School'] != mergeTeamDF['School_opp']]
    
        # getting score differentials and point totals
        mergeTeamDF['scoreDiff'] = mergeTeamDF['Points'] - mergeTeamDF['Points_opp']
        mergeTeamDF['pointTotal'] = mergeTeamDF['Points'] + mergeTeamDF['Points_opp']

        # adding a win column to show if the given team won a game or not
        mergeTeamDF['Win'] = mergeTeamDF['scoreDiff'] > 0
    
        # setting the dataframe to the merged version
        teamDF = mergeTeamDF
    
        teamDF = teamDF[teamDF['School'] == team]
        
        tempDict[dfName] = teamDF

    # grabbing all of the rolling columns that we will have to update (not including opposing columns)
    rollingCols = [col for col in teamDict['Florida State'].columns if 'rolling_sum' in col and '_opp' not in col]
    
    import copy
    
    newTeamDict = copy.deepcopy(teamDict)
    newFCSDict = copy.deepcopy(fcsDict)

    for dictionary in [newTeamDict, newFCSDict]:
        
        # going through teams in teamDict and appending the recent games to their respective dataframes 
        # and adding in rolling sum, talent, and SP+ data
        for team, teamDF in dictionary.items():
    
            # making sure that team played this past week
            if team in tempDict:
    
                teamDF = teamDF.copy()
                
                # appending the recent game onto the current team dataframe
                teamDF = pd.concat([teamDF, tempDict[team]], ignore_index = True)
    
                # getting rolling sum values
                for col in rollingCols:
                    
                    # grabbing the column and window of games to pull from
                    parts = col.split('_')
                    columnName = parts[2]
        
                    if columnName[-1] == '8':
                        columnName = columnName[:-1]
                        window = 8
                    else:
                        columnName = columnName[:-2]
                        window = 20
                    
                    # grabbing the rolling sum and setting the given column in the most recent row with it
                    rollingSum = customRollingSum(teamDF[columnName], window)
        
                    # correcting yardsPerPass and yardPerRushAttempt columns as we did in p1
                    if col == 'rolling_sum_yardsPerPass20' or col == 'rolling_sum_yardsPerRushAttempt20':
                        rollingSum = rollingSum / 20
        
                    if col == 'rolling_sum_yardsPerPass8' or col == 'rolling_sum_yardsPerRushAttempt8':
                        rollingSum = rollingSum / 8
                        
                    teamDF.at[teamDF.index[-1], col] = rollingSum.iloc[-1]
    
                    dictionary[team] = teamDF

        spPath = f'/users/blaizelahman/Development/techprojects/CFB Model/SP+ Data/{year}/SP_{year}week{week}.csv'
        spRatings = pd.read_csv(spPath)
    
        # merging opposing team's rolling sum columns
        for team, teamDF in dictionary.items():
    
            if team in tempDict:
    
                teamDF = dictionary[team].copy()
    
                # grabbing most recent game and index
                lastRow = teamDF.iloc[-1]
                lastIndex = teamDF.index[-1]
            
                # grabbing the game id and opponent's name to access the game in their dataframe
                gameID = lastRow['Game Id']
        
                oppName = lastRow['School_opp']
        
                # grabbing the opponent's dataframe based on what dictionary they're in
                if oppName in dictionary:
                    
                    oppDF = dictionary[oppName]
                    
                elif oppName in fcsDict: 
                    oppDF = fcsDict[oppName]
        
                else: 
                    continue
        
                # grabbing the opponent's rolling_sum columns from the game they played against the given team
                oppRow = oppDF[oppDF['Game Id'] == gameID]
        
                if not oppRow.empty:
            
                    # merging the opponent's rolling_sum columns on the row the team plays them
                    for col in rollingCols:
                        teamDF.loc[lastIndex, col + '_opp'] = oppRow.iloc[0][col]
    
                # skipping the talent and SP+ rating step if it's week 1 because this will be done using
                # grabUpcomingYearTalent and grabUpcomingYearSP
                if week == 1:
                    dictionary[team] = teamDF
                    continue
                
                # ensure columns exist and set them as NaN values if not
                if 'talent' not in teamDF.columns:
                    teamDF['talent'] = pd.NA
                if 'SP' not in teamDF.columns:
                    teamDF['SP'] = pd.NA
                
                # assigning talent from the previous row
                teamDF.loc[lastIndex, 'talent'] = teamDF.loc[lastIndex - 1, 'talent']

                # grabbing team SP rating
                sp = spRatings.loc[spRatings['Team'] == team, 'Rating']

                if len(sp.values) == 0:
                    print(f'No SP+ rating for {team}')
                    dictionary[team] = teamDF
                    continue
                    
                
                teamDF.loc[lastIndex, 'SP'] = sp.values[0]
                
                dictionary[team] = teamDF
    
        # adding in the opposing team's talent and SP+ rating
        for team, teamDF in dictionary.items():
    
            if team in tempDict:
            
                teamDF = dictionary[team].copy()
    
                # grabbing most recent game and index
                lastRow = teamDF.iloc[-1]
                lastIndex = teamDF.index[-1]
        
                oppName = lastRow['School_opp']
        
                # grabbing the opponent's dataframe based on what dictionary they're in
                if oppName in teamDict:
                    
                    oppDF = teamDict[oppName]
        
                    # setting if a team is in the FCS or not because the naming conventions for columns differs
                    fcs = False
                
                elif oppName in fcsDict and oppName not in teamDict: 
                    
                    oppDF = fcsDict[oppName]
        
                    # setting if a team is in the FCS or not because the naming conventions for columns differs
                    fcs = True
        
                else: 
                    continue
        
                # skipping the talent and SP+ adding process if it's week 1
                if week == 1:
                    dictionary[team] = teamDF
                    continue
    
                # seeing if a team is in the FCS or not because the naming conventions for columns differs
                if fcs == False:
                    
                    # grabbing the opposing team's most recent played game to grab their talent and SP+ data
                    filteredOppDF = oppDF[oppDF['Year'] == year]
        
                if fcs == True:
                    filteredOppDF = oppDF[oppDF['Year'] == str(year)]
        
                # checking if dataframe is empty
                if filteredOppDF.empty:
                    print(f"No data for opponent in year {year}: {oppName}")
                    continue 
        
                oppRow = filteredOppDF.iloc[0]
    
                # ensure columns exist and set them as NaN values if not
                if 'talent' not in oppRow.index:
                    oppRow['talent'] = pd.NA
                if 'SP' not in oppRow.index:
                    oppRow['SP'] = pd.NA
                    
                # adding oppossing team talent and SP+ ratings
                for col in ['talent', 'SP']:
                    teamDF.loc[lastIndex, col + '_opp'] = oppRow[col]
        
                dictionary[team] = teamDF


    # now adding in betting data
    # creating Chrome driver
    driver = webdriver.Chrome(service = Service(path), options = chromeOptions)
    # setting the link that directs to the betting data
    bettingDataLink = f'https://collegefootballdata.com/exporter/lines?year={year}&week={week}&seasonType=regular'
    
    driver.get(bettingDataLink)
    time.sleep(4) 
            
    # clicking the query button
    query = driver.find_element(By.XPATH, "//button[contains(span/text(), 'Query')]")
    query.click()
    time.sleep(3) 
            
    # clicking the export button
    export = driver.find_element(By.XPATH, "//button[contains(span/text(), 'Export')]")
    export.click()
    time.sleep(3)

    key = str(year)
            
    # grabs files from CFBData folder
    files = os.listdir(directory)
        
    # grab the file paths for all files ending in .csv
    filePaths = [os.path.join(directory, name) for name in files if name.endswith('.csv')]

    # grabbing the most recently made file out of those in paths
    file = max(filePaths, key=os.path.getctime)
            
    # loading csv file
    bettingData = pd.read_csv(file)

    driver.quit()

    # setting our preferred line providers
    lineProviders = ['EPSN Bet', 'DraftKings', 'consensus', 'Bovada']

    # getting the preferred line for each individual game
    preferredLines = bettingData.groupby('Id').apply(lambda x: getPreferredLine(x, lineProviders)).reset_index(drop = True)

    # merging the preferred lines with each dataframe in teamDict
    for team, teamDF in newTeamDict.items():

        if team in tempDict:

            teamDF = teamDF.copy()

            # getting the last row of the dataframe
            lastRow = teamDF.iloc[-1]
            lastGameId = lastRow['Game Id']
    
            # finding the matching game row in preferredLines
            gameRow = preferredLines[preferredLines['Id'] == lastGameId]
    
            if not gameRow.empty:
                
                gameRow = gameRow.iloc[0] 

                if gameRow['HomeTeam'] == team:
                    gameRow['Spread'] = float(gameRow['Spread']) * -1

                # getting betting columns to update
                cols = teamDF.columns[-13:]
    
                # updating the betting columns in the last row with betting data
                teamDF.loc[teamDF.index[-1], cols] = gameRow[cols].values

            newTeamDict[team] = teamDF

    return [newTeamDict, newFCSDict]
    


In [ ]:
newDicts = grabLastWeekData(2024, 16, teamDict, fcsDict)

In [ ]:
grabUpcomingYearSP(2024, newDicts[0])

In [ ]:
newDicts[0]['Texas'][-5:]

In [ ]:
newDicts[0]["Texas State"][-5:]

In [ ]:
teamDict['Texas State'][-2:]

In [ ]:
for team, df in newDicts[0].items():
    dfCopy = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    newDicts[0][team] = dfCopy
    
    

In [ ]:
for key, team in newDicts[0].items():
    
    name = key.replace(' ', '_') + '_2024week16.csv'
    path = os.path.join('/users/blaizelahman/Development/techprojects/CFB Model/2024 Season/Week 16', name)
    team.to_csv(path)
    print('CSV: ' + name)

In [ ]:
for key, team in newDicts[1].items():
    
    name = key.replace(' ', '_') + '_2024week15csv'
    path = os.path.join('/users/blaizelahman/Desktop/CFB Model/2024 Season/Week 15 FCS', name)
    team.to_csv(path)
    print('CSV: ' + name)

In [14]:
def savePredictions(predsDict):

    predsDF = pd.DataFrame()

    # going through predictions and adding them onto predsDF
    for team, teamPreds in predsDict.items():
        
        # only adding games that were played
        if len(teamPreds) == 7:
            teamDF = pd.DataFrame([teamPreds], columns=['pred', 'spread', 'spreadDiff', 'cover', 'gameID', 'team', 'oppTeam'])
            predsDF = pd.concat([predsDF, teamDF], ignore_index = True)

    return predsDF

In [ ]:
predsDF = savePredictions(predsDict)

In [ ]:
name = 'preds_2024w2_wed.csv'
path = os.path.join('/users/blaizelahman/Desktop/CFB Model/2024 Season/Predictions/Week 2', name)
predsDF.to_csv(path)
print('CSV: ' + name)

In [15]:
def evaluatePreds(week, binData, teamDict, fcsDict):
    
    directory = f'/users/blaizelahman/Development/techprojects/CFB Model/2024 Season/Predictions/Week {week}'
    pattern = os.path.join(directory, f'*_2024w{week}*.csv')
    
    teamFiles = glob.glob(pattern)
    
    predsDict = {}
    
    for file in teamFiles:
        predDF = pd.read_csv(file)
        key = file[-7:-4]
        predDF['day'] = key
        predsDict[key] = predDF
        print(f'Added: {key}')

    predsDF = pd.concat(predsDict.values(), ignore_index=True)

    fbsOnly = []

    for index, game in predsDF.iterrows():
        pred = game['cover']
        spreadDiff = game['spreadDiff']
        gameID = game['gameID']
        team = game['team']

        if team == 'Appalachian State':
            team = 'App State'
        elif team == 'Louisiana Monroe':
            team = 'UL Monroe'
        elif team == 'Connecticut':
            team = 'UConn'
        elif team == 'UT San Antonio':
            team = 'UTSA'
        elif team == 'UMass':
            team = 'Massachusetts'
        elif team == 'Southern Mississippi':
            team = 'Southern Miss'
        
        teamDF = teamDict[team]
        oppTeam = game['oppTeam']

        if oppTeam in fcsDict or oppTeam not in teamDict:
            fbsOnly.append(False)
        else:
            fbsOnly.append(True)

        if gameID in teamDF['Id'].values:
            gameRow = teamDF.loc[teamDF['Id'] == gameID]
            spread = gameRow['Spread'].values[0]
            scoreDiff = gameRow['scoreDiff'].values[0]

            cover = 1 if scoreDiff > spread else -1 if scoreDiff < spread else 0
            result = 0 if cover == 0 else 1 if pred == cover else -1

            predsDF.at[index, 'scoreDiff'] = scoreDiff
            predsDF.at[index, 'result'] = result

            predBin = binData[(binData['lowerBin'] <= spreadDiff) & (binData['upperBin'] > spreadDiff)]
            if not predBin.empty:
                successRate = predBin.iloc[0]['successRate']
                predsDF.at[index, 'successRate'] = successRate
            else:
                print(f'Could not find predBin for {team} game.')
        else:
            print(f'Could not find game id {gameID} in {team} dataframe.')

    # Drop rows where 'successRate' is NaN
    cleanPreds = predsDF.dropna(subset=['successRate'])

    # Group by 'gameID' and find the index of the row with the highest 'successRate' for each gameID
    maxSuccessRate = cleanPreds.groupby('gameID')['successRate'].idxmax()

    # Use these indices to filter cleanPreds
    filteredPredsDF = cleanPreds.loc[maxSuccessRate]

    # Convert fbsOnly to a boolean mask for filtering
    fbsOnly_mask = pd.Series(fbsOnly, index=predsDF.index)
    
    # Filter fbsOnly_mask to match the index of filteredPredsDF
    filtered_fbsOnly_mask = fbsOnly_mask.loc[filteredPredsDF.index]
    
    # Apply the filtered mask to filteredPredsDF
    filteredPredsCopy = filteredPredsDF[filtered_fbsOnly_mask]

    # Calculate win rate
    winRate = len(filteredPredsCopy[filteredPredsCopy['result'] == 1]) / len(filteredPredsCopy)
    regWins = len(filteredPredsCopy[filteredPredsCopy['result'] == 1])
    regLosses = len(filteredPredsCopy) - len(filteredPredsCopy[filteredPredsCopy['result'] == 1])
    print(f'Overall win rate: {winRate}, {regWins}-{regLosses}')

    goodBets = filteredPredsCopy[filteredPredsCopy['successRate'] >= 0.595]
    
    goodWinRate = len(goodBets[goodBets['result'] == 1]) / len(goodBets)
    goodWins = len(goodBets[goodBets['result'] == 1])
    goodLosses = len(goodBets[goodBets['result'] == -1])

    print(f'Win rate on good bets: {goodWinRate}, {goodWins}-{goodLosses}')

    greatBets = filteredPredsCopy[filteredPredsCopy['successRate'] >= 0.645]
    
    greatWinRate = len(greatBets[greatBets['result'] == 1]) / len(greatBets)
    greatWins = len(greatBets[greatBets['result'] == 1])
    greatLosses = len(greatBets[greatBets['result'] == -1])

    print(f'Win rate on great bets: {greatWinRate}, {greatWins}-{greatLosses}')

    bestBets = filteredPredsCopy[filteredPredsCopy['successRate'] >= 0.695]

    if len(bestBets) != 0:
        
        bestWinRate = len(bestBets[bestBets['result'] == 1]) / len(bestBets)
        bestWins = len(bestBets[bestBets['result'] == 1])
        bestLosses = len(bestBets[bestBets['result'] == -1])

        print(f'Win rate on best bets: {bestWinRate}, {bestWins}-{bestLosses}')

    else:
        print('No best bets this week')

    filteredPredsCopy = filteredPredsCopy.drop(columns = ['Unnamed: 0'])

    name = f'results_2024w{week}.csv'
    path = os.path.join('/users/blaizelahman/Development/techprojects/CFB Model/2024 Season/Results', name)
    filteredPredsCopy.to_csv(path)
    print('CSV: ' + name)

    return[filteredPredsCopy, predsDF]

In [ ]:
newDicts[0]['Georgia Tech'][-5:]

In [ ]:
predsCopy, totalPreds = evaluatePreds(15, binData, newDicts[0], newDicts[1])

In [ ]:
predsCopy

In [ ]:
grouped = predsCopy.groupby('day').agg(
    total_games=('result', 'size'),  # Total number of games per day
    win_rate=('result', lambda x: (x == 1).mean())  # Proportion of games with result = 1
)

print(grouped)

filtered_predsCopy = predsCopy[predsCopy['successRate'] >= 0.595]

filtered_grouped = filtered_predsCopy.groupby('day').agg(
    good_games=('result', 'size'),  # Total number of games per day with successRate >= 0.595
    win_rate=('result', lambda x: (x == 1).mean())  # Proportion of games with result = 1
)

print(filtered_grouped)


In [ ]:
newDicts[0]['Duke'][-5:]

In [25]:
def evaluateSeason(year):
    
    directory = f'/users/blaizelahman/Development/techprojects/CFB Model/{year} Season/Results'
    pattern = os.path.join(directory, f'*results_*.csv')
    teamFiles = glob.glob(pattern)
    
    resultsDict = {}
    
    for file in teamFiles:
        
        resultsDF = pd.read_csv(file)
        key_match = re.findall(r"results_2024w(\d+)\.csv", file)
        if key_match:
            week = int(key_match[0])
            resultsDF['week'] = week
            resultsDict[f"w{week}"] = resultsDF
            print(f"Added week {week}")
        else:
            print(f"Skipped non-week file: {file}")

    resultsDF = pd.concat(resultsDict.values(), ignore_index = True)

    # Calculate win rate
    winRate = len(resultsDF[resultsDF['result'] == 1]) / len(resultsDF)
    regWins = len(resultsDF[resultsDF['result'] == 1])
    regLosses = len(resultsDF) - len(resultsDF[resultsDF['result'] == 1])
    print(f'Overall win rate: {winRate}, {regWins}-{regLosses}')

    goodBets = resultsDF[resultsDF['successRate'] >= 0.595]
    
    goodWinRate = len(goodBets[goodBets['result'] == 1]) / len(goodBets)
    goodWins = len(goodBets[goodBets['result'] == 1])
    goodLosses = len(goodBets[goodBets['result'] == -1])

    print(f'Win rate on good bets: {goodWinRate}, {goodWins}-{goodLosses}')

    greatBets = resultsDF[resultsDF['successRate'] >= 0.645]
    
    greatWinRate = len(greatBets[greatBets['result'] == 1]) / len(greatBets)
    greatWins = len(greatBets[greatBets['result'] == 1])
    greatLosses = len(greatBets[greatBets['result'] == -1])

    print(f'Win rate on great bets: {greatWinRate}, {greatWins}-{greatLosses}')

    bestBets = resultsDF[resultsDF['successRate'] >= 0.695]

    if len(bestBets) != 0:
        
        bestWinRate = len(bestBets[bestBets['result'] == 1]) / len(bestBets)
        bestWins = len(bestBets[bestBets['result'] == 1])
        bestLosses = len(bestBets[bestBets['result'] == -1])

        print(f'Win rate on best bets: {bestWinRate}, {bestWins}-{bestLosses}')

    else:
        print('No best bets this week')

    return resultsDF

In [26]:
resultsDF = evaluateSeason(2024)

Added week 15
Added week 14
Added week 9
Added week 10
Added week 11
Added week 8
Added week 13
Added week 12
Added week 6
Added week 7
Added week 5
Added week 4
Added week 1
Added week 3
Added week 2
Skipped non-week file: /users/blaizelahman/Development/techprojects/CFB Model/2024 Season/Results/results_2024_reg_season.csv
Overall win rate: 0.47161572052401746, 324-363
Win rate on good bets: 0.48231511254019294, 150-156
Win rate on great bets: 0.45714285714285713, 48-56
Win rate on best bets: 0.5384615384615384, 7-6


In [27]:
name = f'results_2024_reg_season.csv'    
path = os.path.join('/users/blaizelahman/Development/techprojects/CFB Model/2024 Season/Results', name)
resultsDF.to_csv(path)

In [28]:
for week in range(14):

    tempDF = resultsDF[resultsDF['week'] >= week]

    print(f'WIN RATE SINCE WEEK {week} \n')
    
    # Calculate win rate
    winRate = len(tempDF[tempDF['result'] == 1]) / len(tempDF)
    regWins = len(tempDF[tempDF['result'] == 1])
    regLosses = len(tempDF) - len(tempDF[tempDF['result'] == 1])
    print(f'Overall win rate: {winRate}, {regWins}-{regLosses}')

    goodBets = tempDF[tempDF['successRate'] >= 0.595]
    
    goodWinRate = len(goodBets[goodBets['result'] == 1]) / len(goodBets)
    goodWins = len(goodBets[goodBets['result'] == 1])
    goodLosses = len(goodBets[goodBets['result'] == -1])

    print(f'Win rate on good bets: {goodWinRate}, {goodWins}-{goodLosses}')

    greatBets = tempDF[tempDF['successRate'] >= 0.645]
    
    greatWinRate = len(greatBets[greatBets['result'] == 1]) / len(greatBets)
    greatWins = len(greatBets[greatBets['result'] == 1])
    greatLosses = len(greatBets[greatBets['result'] == -1])

    print(f'Win rate on great bets: {greatWinRate}, {greatWins}-{greatLosses}')

    bestBets = tempDF[tempDF['successRate'] >= 0.695]

    if len(bestBets) != 0:
        
        bestWinRate = len(bestBets[bestBets['result'] == 1]) / len(bestBets)
        bestWins = len(bestBets[bestBets['result'] == 1])
        bestLosses = len(bestBets[bestBets['result'] == -1])

        print(f'Win rate on best bets: {bestWinRate}, {bestWins}-{bestLosses}')

    else:
        print('No best bets this week')

    print()

WIN RATE SINCE WEEK 0 

Overall win rate: 0.47161572052401746, 324-363
Win rate on good bets: 0.48231511254019294, 150-156
Win rate on great bets: 0.45714285714285713, 48-56
Win rate on best bets: 0.5384615384615384, 7-6

WIN RATE SINCE WEEK 1 

Overall win rate: 0.47161572052401746, 324-363
Win rate on good bets: 0.48231511254019294, 150-156
Win rate on great bets: 0.45714285714285713, 48-56
Win rate on best bets: 0.5384615384615384, 7-6

WIN RATE SINCE WEEK 2 

Overall win rate: 0.4732824427480916, 310-345
Win rate on good bets: 0.48299319727891155, 142-147
Win rate on great bets: 0.46464646464646464, 46-52
Win rate on best bets: 0.5833333333333334, 7-5

WIN RATE SINCE WEEK 3 

Overall win rate: 0.4704918032786885, 287-323
Win rate on good bets: 0.4833948339483395, 131-135
Win rate on great bets: 0.4731182795698925, 44-48
Win rate on best bets: 0.5833333333333334, 7-5

WIN RATE SINCE WEEK 4 

Overall win rate: 0.4734042553191489, 267-297
Win rate on good bets: 0.488, 122-123
Win rate

In [29]:
grouped = resultsDF.groupby('day').agg(
    total_games=('result', 'size'),  # Total number of games per day
    proportion_result_1=('result', lambda x: (x == 1).mean())  # Proportion of games with result = 1
)

print(grouped)

filtered_resultsDF = resultsDF[resultsDF['successRate'] >= 0.595]

filtered_grouped = filtered_resultsDF.groupby('day').agg(
    total_games=('result', 'size'),  # Total number of games per day with successRate >= 0.595
    proportion_result_1=('result', lambda x: (x == 1).mean())  # Proportion of games with result = 1
)

print(filtered_grouped)

filtered_bigSpread = filtered_resultsDF[resultsDF['spread'].abs() >= 27.5]

filtered_grouped = filtered_bigSpread.agg(
    total_games=('result', 'size'),  # Total number of games per day with successRate >= 0.595
    proportion_result_1=('result', lambda x: (x == 1).mean())  # Proportion of games with result = 1
)

print(filtered_grouped)

onlyGood = resultsDF[resultsDF['successRate'] >= 0.595]

onlyGood = onlyGood[onlyGood['successRate'] <= 0.65]

filtered_grouped = onlyGood.agg(
    total_games=('result', 'size'),  # Total number of games per day with successRate >= 0.595
    proportion_result_1=('result', lambda x: (x == 1).mean())  # Proportion of games with result = 1
)

print(filtered_grouped)

     total_games  proportion_result_1
day                                  
fri          153             0.477124
mon          109             0.541284
sun           56             0.446429
thu           69             0.478261
tue          134             0.470149
wed          134             0.425373
     total_games  proportion_result_1
day                                  
fri           70             0.500000
mon           42             0.619048
sun           22             0.272727
thu           32             0.531250
tue           61             0.508197
wed           67             0.402985
                        result
total_games          22.000000
proportion_result_1   0.318182
                         result
total_games          232.000000
proportion_result_1    0.491379


/var/folders/1v/_zpppqpd4gv8l0z9qkhwhh1h0000gn/T/ipykernel_94462/343575672.py:17: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_bigSpread = filtered_resultsDF[resultsDF['spread'].abs() >= 27.5]


In [30]:
def runModel(year, week, day):

    directory = f'/users/blaizelahman/Development/techprojects/CFB Model/{year} Season/Week {week-1}'
    pattern = os.path.join(directory, f'*_{year}week{week-1}*.csv')
    
    teamFiles = glob.glob(pattern)
    
    teamDict = {}
    
    for file in teamFiles:
    
        teamDF = pd.read_csv(file)
    
        if teamDF.shape[0] >= 56:

            if teamDF['School'][0] == 'Appalachian State' or teamDF['School'][0] == 'App State':
                key = 'App State'
            elif teamDF['School'][0] == 'UMass' or teamDF['School'][0] == 'Massachusetts':
                key = 'Massachusetts'
            elif teamDF['School'][0] == 'Louisiana Monroe' or teamDF['School'][0] == 'UL Monroe':
                key = 'UL Monroe'
            elif teamDF['School'][0] == 'Connecticut' or teamDF['School'][0] == 'UConn':
                key = 'UConn'
            elif teamDF['School'][0] == 'UT San Antonio' or teamDF['School'][0] == 'UTSA':
                key = 'UTSA'
            elif teamDF['School'][0] == 'Southern Mississippi' or teamDF['School'][0] == 'Southern Miss':
                key = 'Southern Miss'
            else:
                key = teamDF['School'][0]
                
            teamDict[key] = teamDF
        
            print(f'Added: {key}')

    directory = f'/users/blaizelahman/Development/techprojects/CFB Model/{year} Season/Week {week-1} FCS'
    pattern = os.path.join(directory, f'*_{year}week{week-1}*.csv')
    
    teamFiles = glob.glob(pattern)
    
    fcsDict = {}
    
    for file in teamFiles:
    
        teamDF = pd.read_csv(file)
    
        if teamDF.shape[0] >= 10:
        
            key = teamDF['School'][0]
            fcsDict[key] = teamDF
        
        else:
            print(f'{team} not added.')

    grabUpcomingWeekData(year, week, teamDict, fcsDict)

    features = [col for col in teamDict['Florida State'].columns if any(word in col for word in ['rolling_sum','talent','SP'])]

    predsDict = predictUpcomingWeek(modelDict, teamDict, features)

    path = '/users/blaizelahman/Development/techprojects/CFB Model/Bin Data/Bin_Data.csv'
    binData = pd.read_csv(path)

    print()

    printPredictions(predsDict, binData, teamDict, fcsDict)

    predsDF = savePredictions(predsDict)

    name = f'preds_{year}w{week}_{day}.csv'
    path = os.path.join(f'/users/blaizelahman/Development/techprojects/CFB Model/{year} Season/Predictions/Week {week}', name)
    predsDF.to_csv(path)
    print('CSV: ' + name)

    return teamDict
    

In [ ]:
dayDict = runModel(2024, 17, 'dec20')

In [ ]:
dayDict['Indiana'][-10:]

In [ ]:
dayDict['Utah State'][-10:]